# 06 -- Human-judgment correlation

Every automated metric so far (ROUGE, entity-F1, NLI fact-consistency) is a proxy for what a person would actually think of the note. This closes that gap: sample outputs, get a human to rate them, check how well the automated reward actually tracks a real opinion.

Needs a merged model -- run `07_merge_and_export.ipynb` first, or point `--model-path` at the raw `dpo_model` adapter dir if you'd rather not merge yet (works the same, just slower to load).

*(Uses the same `GITHUB_TOKEN` Colab secret set up in notebook 01 -- see that notebook if you haven't set it up yet.)*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/MedAlignRL'
GITHUB_USERNAME = 'YOUR_USERNAME'   # <-- change this
GITHUB_REPO = 'MedAlignRL'         # <-- change if you named it differently

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
    print("No GITHUB_TOKEN secret found. Fine if your repo is public -- if it's "
          "private this clone will fail. See the setup note above.")

if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
else:
    REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'

if not os.path.exists(PROJECT_DIR):
    print("Cloning into Drive (first time)...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already in Drive, pulling latest...")
    !cd {PROJECT_DIR} && git remote set-url origin {REPO_URL} && git fetch origin && git reset --hard origin/main

%cd {PROJECT_DIR}
!pip install -q -U -r requirements-colab.txt
!pip uninstall -y -q torchao  # Colab preinstalls an old torchao; peft raises ImportError on it during LoRA dispatch, and this repo never uses it
!pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz  # reward.py's clinical NER needs this in every notebook that scores rewards, not just notebook 01

# en_core_sci_sm 0.5.4's config.cfg stores include_static_vectors as the
# string "False" (an old spaCy 3.7 serialization quirk); spaCy 3.8's
# stricter config validator requires an actual bool, so loading fails with
# a Config error otherwise. Patch it in place.
import importlib.util, pathlib
spec = importlib.util.find_spec("en_core_sci_sm")
for cfg in pathlib.Path(spec.origin).parent.rglob("config.cfg"):
    text = cfg.read_text()
    fixed = text.replace('include_static_vectors = "False"', 'include_static_vectors = false')
    if fixed != text:
        cfg.write_text(fixed)
        print(f"Patched {cfg}")


In [ ]:
!nvidia-smi

### Step 1: generate the rating sheet

In [ ]:
%cd {PROJECT_DIR}/src
!python human_eval.py --mode generate --model-path ../outputs/dpo_model_merged --n 30

### Step 2: rate it

Open `outputs/human_eval_sheet.csv` (it's on Drive, so you can open it directly from Drive's web UI, or download it, edit locally, and re-upload to the same path). Fill in `expert_rating` with a 1-5 score per row: does every stated fact actually trace back to something in the dialogue? Leave rows blank if you're skipping them.

If you can get someone with actual clinical background to do this (a med student, a nurse, a doctor friend) the number means a lot more than if you rate it yourself -- worth a text to someone if you know anyone.

### Step 3: score it

In [ ]:
!python human_eval.py --mode score